In [ ]:
!pip install torch torchvision torchaudio transformers scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_scheduler
from tqdm import tqdm
from torch.optim import AdamW
import os
import json
from datetime import datetime

In [ ]:
# ============================================================================
# GPU/CUDA SETUP AND VERIFICATION
# ============================================================================
print("="*80)
print("GPU/CUDA SETUP AND VERIFICATION")
print("="*80)

# Check CUDA availability
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Set device to GPU
    device = torch.device('cuda')
    print(f"\n✓ Using GPU: {torch.cuda.get_device_name(0)}")

    # Clear GPU cache
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared")
else:
    device = torch.device('cpu')
    print("\n⚠ CUDA not available. Using CPU.")
    print("Note: Training will be significantly slower on CPU.")

print("="*80)

GPU/CUDA SETUP AND VERIFICATION

PyTorch Version: 2.9.0+cu126
CUDA Available: True
CUDA Version: 12.6
Number of GPUs: 1
Current GPU: 0
GPU Name: Tesla T4
GPU Memory: 14.74 GB

✓ Using GPU: Tesla T4
✓ GPU cache cleared


In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================

class Config:
    # Paths
    DATA_PATH = "/content/drive/MyDrive/fyp1_ambiguity_data/shuffled_ambiguity_fyp1.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/FYP_ambiguity"
    CHECKPOINT_DIR = "/content/drive/MyDrive/FYP_checkpoints_ambiguity"

    # Model
    MODEL_NAME = "roberta-base"
    MAX_LENGTH = 128

    # Training
    BATCH_SIZE = 16
    NUM_EPOCHS = 4
    LEARNING_RATE = 2e-5
    WARMUP_STEPS = 0

    # Validation
    VAL_SIZE = 0.1  # 10% of training data for validation
    TEST_SIZE = 0.2

    # Checkpoint
    SAVE_EVERY_N_STEPS = 100
    SAVE_BEST_ONLY = True

    RANDOM_STATE = 42

config = Config()

# Create directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================================
# CELL 3: LOAD AND CLEAN DATA
# ============================================================================

print("="*80)
print("STEP 2: LOADING AND PREPROCESSING DATA")
print("="*80)

# Load dataset
df = pd.read_csv(config.DATA_PATH)

print("\n" + "="*80)
print("RAW DATA INFO")
print("="*80)
print(f"Total samples before cleaning: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# DATA CLEANING
print("\n" + "="*80)
print("DATA CLEANING")
print("="*80)

# Remove rows with NaN in critical columns
critical_columns = ['requirement_text', 'is_ambiguous', 'lexical_ambiguity',
                   'syntactic_ambiguity', 'semantic_ambiguity',
                   'pragmatic_ambiguity', 'referential_ambiguity', 'ambiguity_score']

df_clean = df.dropna(subset=critical_columns)

# Remove any empty text rows
df_clean = df_clean[df_clean['requirement_text'].str.strip() != '']

# Convert to appropriate types
df_clean['is_ambiguous'] = df_clean['is_ambiguous'].astype(int)
df_clean['lexical_ambiguity'] = df_clean['lexical_ambiguity'].astype(int)
df_clean['syntactic_ambiguity'] = df_clean['syntactic_ambiguity'].astype(int)
df_clean['semantic_ambiguity'] = df_clean['semantic_ambiguity'].astype(int)
df_clean['pragmatic_ambiguity'] = df_clean['pragmatic_ambiguity'].astype(int)
df_clean['referential_ambiguity'] = df_clean['referential_ambiguity'].astype(int)
df_clean['ambiguity_score'] = df_clean['ambiguity_score'].astype(float)

print(f"Rows removed: {len(df) - len(df_clean)}")
print(f"Total samples after cleaning: {len(df_clean)}")

# Reset index
df = df_clean.reset_index(drop=True)

print("\n" + "="*80)
print("CLEANED DATA - FIRST 10 ROWS")
print("="*80)
print(df.head(10))

print("\n" + "="*80)
print("DATASET STATISTICS")
print("="*80)
print(f"\nAmbiguity distribution (for training):")
print(df['is_ambiguous'].value_counts())
print(f"\nAmbiguity types distribution (for score calculation only):")
print(f"Lexical: {df['lexical_ambiguity'].sum()}")
print(f"Syntactic: {df['syntactic_ambiguity'].sum()}")
print(f"Semantic: {df['semantic_ambiguity'].sum()}")
print(f"Pragmatic: {df['pragmatic_ambiguity'].sum()}")
print(f"Referential: {df['referential_ambiguity'].sum()}")
print(f"\nAmbiguity score statistics:")
print(df['ambiguity_score'].describe())

STEP 2: LOADING AND PREPROCESSING DATA

RAW DATA INFO
Total samples before cleaning: 2775
Columns: ['req_id', 'requirement_text', 'is_ambiguous', 'lexical_ambiguity', 'syntactic_ambiguity', 'semantic_ambiguity', 'pragmatic_ambiguity', 'referential_ambiguity', 'ambiguity_score']

Missing values per column:
req_id                   103
requirement_text         103
is_ambiguous             103
lexical_ambiguity        103
syntactic_ambiguity      103
semantic_ambiguity       103
pragmatic_ambiguity      103
referential_ambiguity    103
ambiguity_score          103
dtype: int64

DATA CLEANING
Rows removed: 103
Total samples after cleaning: 2672

CLEANED DATA - FIRST 10 ROWS
   req_id                                   requirement_text  is_ambiguous  \
0   899.0  The wildlife rehabilitation database should tr...             1   
1  1505.0  The intrusion detection system shall analyze n...             0   
2  1063.0  The user-facing metrics must be easy to unders...             1   
3  1942.0

In [ ]:
# ============================================================================
# CELL 4: SPLIT DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 3: SPLITTING DATA")
print("="*80)

# First split: separate test set (stratified on is_ambiguous only)
train_val_df, test_df = train_test_split(
    df,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=df['is_ambiguous']  # Balanced split on ambiguous/not ambiguous
)

# Second split: separate validation set from training
train_df, val_df = train_test_split(
    train_val_df,
    test_size=config.VAL_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=train_val_df['is_ambiguous']  # Balanced split on ambiguous/not ambiguous
)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

print(f"\nTraining set ambiguity distribution:")
print(train_df['is_ambiguous'].value_counts())
print(f"\nValidation set ambiguity distribution:")
print(val_df['is_ambiguous'].value_counts())
print(f"\nTest set ambiguity distribution:")
print(test_df['is_ambiguous'].value_counts())


STEP 3: SPLITTING DATA
Training set size: 1923
Validation set size: 214
Test set size: 535

Training set ambiguity distribution:
is_ambiguous
1    962
0    961
Name: count, dtype: int64

Validation set ambiguity distribution:
is_ambiguous
0    107
1    107
Name: count, dtype: int64

Test set ambiguity distribution:
is_ambiguous
0    268
1    267
Name: count, dtype: int64


In [ ]:
# ============================================================================
# CELL 5: DATASET CLASS
# ============================================================================

class AmbiguityDataset(Dataset):
    """
    Dataset for ambiguity detection.
    - Model trains on: is_ambiguous (binary classification)
    - Additional info stored: ambiguity types and scores (for analysis/score calculation)
    """
    def __init__(self, texts, labels, ambiguity_scores, ambiguity_types, tokenizer, max_length):
        self.texts = texts
        self.labels = labels  # is_ambiguous (0 or 1)
        self.ambiguity_scores = ambiguity_scores
        self.ambiguity_types = ambiguity_types  # Dict with lexical, syntactic, etc.
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Binary: ambiguous or not
        ambiguity_score = self.ambiguity_scores[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),  # Only this is used for training
            'ambiguity_score': torch.tensor(ambiguity_score, dtype=torch.float),
            'ambiguity_types': torch.tensor([
                self.ambiguity_types['lexical'][idx],
                self.ambiguity_types['syntactic'][idx],
                self.ambiguity_types['semantic'][idx],
                self.ambiguity_types['pragmatic'][idx],
                self.ambiguity_types['referential'][idx]
            ], dtype=torch.float)
        }

print("\n" + "="*80)
print("STEP 4: DATASET CLASS DEFINED")
print("="*80)
print("Note: Model trains ONLY on 'is_ambiguous' (binary)")
print("Ambiguity types are stored for score calculation and explanation")


STEP 4: DATASET CLASS DEFINED
Note: Model trains ONLY on 'is_ambiguous' (binary)
Ambiguity types are stored for score calculation and explanation


In [ ]:
# ============================================================================
# CELL 6: TOKENIZER AND DATASETS
# ============================================================================

print("\n" + "="*80)
print("STEP 5: INITIALIZING TOKENIZER AND CREATING DATASETS")
print("="*80)

tokenizer = RobertaTokenizer.from_pretrained(config.MODEL_NAME)

# Prepare ambiguity types dictionaries
train_ambiguity_types = {
    'lexical': train_df['lexical_ambiguity'].values,
    'syntactic': train_df['syntactic_ambiguity'].values,
    'semantic': train_df['semantic_ambiguity'].values,
    'pragmatic': train_df['pragmatic_ambiguity'].values,
    'referential': train_df['referential_ambiguity'].values
}

val_ambiguity_types = {
    'lexical': val_df['lexical_ambiguity'].values,
    'syntactic': val_df['syntactic_ambiguity'].values,
    'semantic': val_df['semantic_ambiguity'].values,
    'pragmatic': val_df['pragmatic_ambiguity'].values,
    'referential': val_df['referential_ambiguity'].values
}

test_ambiguity_types = {
    'lexical': test_df['lexical_ambiguity'].values,
    'syntactic': test_df['syntactic_ambiguity'].values,
    'semantic': test_df['semantic_ambiguity'].values,
    'pragmatic': test_df['pragmatic_ambiguity'].values,
    'referential': test_df['referential_ambiguity'].values
}

# Create datasets
train_dataset = AmbiguityDataset(
    train_df['requirement_text'].values,
    train_df['is_ambiguous'].values,
    train_df['ambiguity_score'].values,
    train_ambiguity_types,
    tokenizer,
    config.MAX_LENGTH
)

val_dataset = AmbiguityDataset(
    val_df['requirement_text'].values,
    val_df['is_ambiguous'].values,
    val_df['ambiguity_score'].values,
    val_ambiguity_types,
    tokenizer,
    config.MAX_LENGTH
)

test_dataset = AmbiguityDataset(
    test_df['requirement_text'].values,
    test_df['is_ambiguous'].values,
    test_df['ambiguity_score'].values,
    test_ambiguity_types,
    tokenizer,
    config.MAX_LENGTH
)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


STEP 5: INITIALIZING TOKENIZER AND CREATING DATASETS
Training dataset size: 1923
Validation dataset size: 214
Test dataset size: 535


In [ ]:
# ============================================================================
# CELL 7: DATA LOADERS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: CREATING DATA LOADERS")
print("="*80)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")


STEP 6: CREATING DATA LOADERS
Number of training batches: 121
Number of validation batches: 14
Number of test batches: 34


In [ ]:
# ============================================================================
# CELL 8: MODEL INITIALIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: INITIALIZING MODEL")
print("="*80)

# Device was already configured in the first cell
print(f"Using device: {device}")

# Initialize model for binary classification (is_ambiguous: 0 or 1)
model = RobertaForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=2  # Binary: ambiguous or not
)

model = model.to(device)

# Enable gradient checkpointing to save GPU memory (optional)
if torch.cuda.is_available():
    model.gradient_checkpointing_enable()
    print("✓ Gradient checkpointing enabled for memory efficiency")

# Print model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

if torch.cuda.is_available():
    print(f"\nGPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

print("\nModel initialized successfully!")
print("Training objective: Binary classification (is_ambiguous)")


STEP 7: INITIALIZING MODEL
Using device: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Gradient checkpointing enabled for memory efficiency

Total parameters: 124,647,170
Trainable parameters: 124,647,170

GPU Memory allocated: 0.55 GB
GPU Memory reserved: 0.60 GB

Model initialized successfully!
Training objective: Binary classification (is_ambiguous)


In [ ]:
# ============================================================================
# CELL 9: OPTIMIZER AND SCHEDULER
# ============================================================================

print("\n" + "="*80)
print("STEP 8: SETTING UP OPTIMIZER AND SCHEDULER")
print("="*80)

optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE)

num_training_steps = len(train_loader) * config.NUM_EPOCHS
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=config.WARMUP_STEPS,
    num_training_steps=num_training_steps
)

# Mixed precision training for faster computation on GPU
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_amp else None

print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {config.WARMUP_STEPS}")
if use_amp:
    print("✓ Mixed precision training (AMP) enabled for faster GPU training")
else:
    print("Mixed precision training disabled (CPU mode)")


STEP 8: SETTING UP OPTIMIZER AND SCHEDULER
Total training steps: 484
Warmup steps: 0
✓ Mixed precision training (AMP) enabled for faster GPU training


/tmp/ipython-input-2399424284.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


In [ ]:
# ============================================================================
# CELL 10: TRAINING AND VALIDATION FUNCTIONS
# ============================================================================

def train_epoch(model, dataloader, optimizer, scheduler, device, scaler=None):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []

    progress_bar = tqdm(dataloader, desc="Training")
    use_amp = scaler is not None

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)  # Only is_ambiguous is used

        optimizer.zero_grad()

        # Mixed precision training
        if use_amp:
            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)

    return avg_loss, accuracy

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)

    return avg_loss, accuracy, predictions, true_labels

print("\n" + "="*80)
print("STEP 9: TRAINING AND VALIDATION FUNCTIONS DEFINED")
print("="*80)


STEP 9: TRAINING AND VALIDATION FUNCTIONS DEFINED


In [ ]:
# ============================================================================
# CELL 11: TRAINING LOOP
# ============================================================================

print("\n" + "="*80)
print("STEP 10: STARTING TRAINING")
print("="*80)

best_val_accuracy = 0
training_stats = []

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"Epoch {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")

    # GPU memory status at start of epoch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU Memory before epoch: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

    # Training
    train_loss, train_accuracy = train_epoch(model, train_loader, optimizer, lr_scheduler, device, scaler)

    print(f"\nTraining Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.4f}")

    # Validation
    val_loss, val_accuracy, val_preds, val_labels = validate(model, val_loader, device)

    print(f"\nValidation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    # GPU memory status after epoch
    if torch.cuda.is_available():
        print(f"\nGPU Memory after epoch: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
        print(f"GPU Memory peak: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")

    # Save statistics
    training_stats.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy
    })

    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        # Save model and tokenizer properly
        print(f"\nSaving model to {config.OUTPUT_DIR}...")
        model.save_pretrained(config.OUTPUT_DIR, safe_serialization=True)
        tokenizer.save_pretrained(config.OUTPUT_DIR)

        # Also save as PyTorch checkpoint for easier loading
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_accuracy,
        }, os.path.join(config.OUTPUT_DIR, 'best_model_checkpoint.pt'))

        print(f"✓ New best model saved with validation accuracy: {val_accuracy:.4f}")

print("\n" + "="*80)
print("TRAINING COMPLETED!")
print("="*80)
print(f"Best validation accuracy: {best_val_accuracy:.4f}")

# Final GPU memory cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"\nFinal GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


STEP 10: STARTING TRAINING

Epoch 1/4
GPU Memory before epoch: 0.55 GB


Training:   0%|          | 0/121 [00:00<?, ?it/s]/tmp/ipython-input-2265291196.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training: 100%|██████████| 121/121 [00:18<00:00,  6.39it/s, loss=0.0046]



Training Loss: 0.1882
Training Accuracy: 0.9194


Validation: 100%|██████████| 14/14 [00:01<00:00, 10.80it/s]



Validation Loss: 0.0297
Validation Accuracy: 0.9953

GPU Memory after epoch: 1.99 GB
GPU Memory peak: 2.47 GB

Saving model to /content/drive/MyDrive/FYP_ambiguity...
✓ New best model saved with validation accuracy: 0.9953

Epoch 2/4
GPU Memory before epoch: 1.99 GB


...


Validation Loss: 0.0240
Validation Accuracy: 0.9953

GPU Memory after epoch: 1.99 GB
GPU Memory peak: 2.47 GB

Epoch 4/4
GPU Memory before epoch: 1.99 GB


Training:   0%|          | 0/121 [00:00<?, ?it/s]/tmp/ipython-input-2265291196.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training: 100%|██████████| 121/121 [00:20<00:00,  5.78it/s, loss=0.00201]



Training Loss: 0.0011
Training Accuracy: 1.0000


Validation: 100%|██████████| 14/14 [00:01<00:00, 10.42it/s]


Validation Loss: 0.0238
Validation Accuracy: 0.9953

GPU Memory after epoch: 1.99 GB
GPU Memory peak: 2.47 GB

TRAINING COMPLETED!
Best validation accuracy: 0.9953

Final GPU Memory: 1.99 GB


In [ ]:
# ============================================================================
# CELL 12: SAVE TRAINING STATISTICS
# ============================================================================

print("\n" + "="*80)
print("STEP 11: SAVING TRAINING STATISTICS")
print("="*80)

stats_df = pd.DataFrame(training_stats)
stats_df.to_csv(os.path.join(config.OUTPUT_DIR, 'training_stats.csv'), index=False)

print("\nTraining Statistics:")
print(stats_df)
print(f"\nStatistics saved to: {os.path.join(config.OUTPUT_DIR, 'training_stats.csv')}")


STEP 11: SAVING TRAINING STATISTICS

Training Statistics:
   epoch  train_loss  train_accuracy  val_loss  val_accuracy
0      1    0.188150        0.919397  0.029728      0.995327
1      2    0.013551        0.996880  0.024784      0.995327
2      3    0.001806        1.000000  0.024048      0.995327
3      4    0.001114        1.000000  0.023820      0.995327

Statistics saved to: /content/drive/MyDrive/FYP_ambiguity/training_stats.csv


In [ ]:
# ============================================================================
# CELL 13: EVALUATION ON TEST SET
# ============================================================================

print("\n" + "="*80)
print("STEP 12: EVALUATING ON TEST SET")
print("="*80)

# Load best model
model = RobertaForSequenceClassification.from_pretrained(config.OUTPUT_DIR)
model = model.to(device)

test_loss, test_accuracy, test_preds, test_labels = validate(model, test_loader, device)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Calculate metrics
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_preds,
    average='binary'
)

print(f"\nTest Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(test_labels, test_preds,
                          target_names=['Not Ambiguous', 'Ambiguous']))


STEP 12: EVALUATING ON TEST SET


Validation: 100%|██████████| 34/34 [00:03<00:00,  8.88it/s]



Test Loss: 0.0207
Test Accuracy: 0.9981

Test Precision: 1.0000
Test Recall: 0.9963
Test F1-Score: 0.9981

CLASSIFICATION REPORT
               precision    recall  f1-score   support

Not Ambiguous       1.00      1.00      1.00       268
    Ambiguous       1.00      1.00      1.00       267

     accuracy                           1.00       535
    macro avg       1.00      1.00      1.00       535
 weighted avg       1.00      1.00      1.00       535



In [ ]:
!pip install sentence-transformers scikit-learn -q

import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import numpy as np
import pickle
import os

In [ ]:
# ============================================================================
# CELL 14: IMPROVED MODEL - FINAL VERSION WITH THRESHOLD CORRECTIONS
# ============================================================================

print("="*80)
print("IMPROVED AI MODEL - Final Version with Threshold Corrections")
print("="*80)

# Paths
DATA_PATH = '/content/drive/MyDrive/fyp1_ambiguity_data/shuffled_ambiguity_fyp1.csv'
MODEL_PATH = "/content/drive/MyDrive/improved_ambiguity_model_2.pkl"
EMBEDDER_NAME = "all-MiniLM-L6-v2"

# Global variables
classifiers = None
thresholds = None
embedder = None
TYPE_NAMES = ['lexical', 'syntactic', 'semantic', 'pragmatic', 'referential']

# ============================================================================
# LOAD OR TRAIN MODEL
# ============================================================================

if os.path.exists(MODEL_PATH):
    print("✓ Loading existing model...")
    with open(MODEL_PATH, 'rb') as f:
        model_data = pickle.load(f)
        classifiers = model_data['classifiers']
        thresholds = model_data['thresholds']
    embedder = SentenceTransformer(EMBEDDER_NAME)
    print("✓ Model loaded!")
    print(f"  Original thresholds: {thresholds}")

else:
    print("Training new model...\n")

    # STEP 1: LOAD DATA
    print("STEP 1: Loading data...")
    df = pd.read_csv(DATA_PATH)
    ambiguous_df = df[df['is_ambiguous'] == 1].copy()
    print(f"Total ambiguous samples: {len(ambiguous_df)}")

    def create_labels(row):
        return [
            int(row['lexical_ambiguity']) if pd.notna(row['lexical_ambiguity']) else 0,
            int(row['syntactic_ambiguity']) if pd.notna(row['syntactic_ambiguity']) else 0,
            int(row['semantic_ambiguity']) if pd.notna(row['semantic_ambiguity']) else 0,
            int(row['pragmatic_ambiguity']) if pd.notna(row['pragmatic_ambiguity']) else 0,
            int(row['referential_ambiguity']) if pd.notna(row['referential_ambiguity']) else 0
        ]

    ambiguous_df['labels'] = ambiguous_df.apply(create_labels, axis=1)

    print("\nOriginal distribution:")
    for i, name in enumerate(['Lexical', 'Syntactic', 'Semantic', 'Pragmatic', 'Referential']):
        count = sum([labels[i] for labels in ambiguous_df['labels']])
        print(f"  {name:<12s}: {count:4d} ({count/len(ambiguous_df)*100:.1f}%)")

    # STEP 2: BALANCE DATA
    print("\nSTEP 2: Balancing data...")

    lexical_df = ambiguous_df[ambiguous_df['labels'].apply(lambda x: x[0] == 1)]
    syntactic_df = ambiguous_df[ambiguous_df['labels'].apply(lambda x: x[1] == 1)]
    semantic_df = ambiguous_df[ambiguous_df['labels'].apply(lambda x: x[2] == 1)]
    pragmatic_df = ambiguous_df[ambiguous_df['labels'].apply(lambda x: x[3] == 1)]
    referential_df = ambiguous_df[ambiguous_df['labels'].apply(lambda x: x[4] == 1)]

    balanced_dfs = []
    balanced_dfs.append(lexical_df.sample(n=min(300, len(lexical_df)), random_state=42))
    balanced_dfs.extend([syntactic_df] * 10)
    balanced_dfs.append(semantic_df.sample(n=min(300, len(semantic_df)), random_state=42))
    balanced_dfs.append(pragmatic_df.sample(n=min(300, len(pragmatic_df)), random_state=42))
    balanced_dfs.extend([referential_df] * 5)

    balanced_df = pd.concat(balanced_dfs, ignore_index=True)
    balanced_df = balanced_df.drop_duplicates(subset=['requirement_text'])
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Balanced size: {len(balanced_df)}")

    # STEP 3: SPLIT
    print("\nSTEP 3: Splitting...")
    train_df, val_df = train_test_split(balanced_df, test_size=0.2, random_state=42)
    print(f"  Train: {len(train_df)}, Val: {len(val_df)}")

    # STEP 4: EMBEDDINGS
    print("\nSTEP 4: Generating embeddings...")
    embedder = SentenceTransformer(EMBEDDER_NAME)

    train_embeddings = embedder.encode(train_df['requirement_text'].tolist(), show_progress_bar=True)
    val_embeddings = embedder.encode(val_df['requirement_text'].tolist(), show_progress_bar=True)

    train_labels = np.array(train_df['labels'].tolist())
    val_labels = np.array(val_df['labels'].tolist())

    # STEP 5: TRAIN
    print("\nSTEP 5: Training classifiers...")

    classifiers = []
    for i, name in enumerate(['Lexical', 'Syntactic', 'Semantic', 'Pragmatic', 'Referential']):
        print(f"  {name}...")

        if name in ['Syntactic', 'Referential']:
            clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=0.5, random_state=42)
        else:
            clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=42)

        clf.fit(train_embeddings, train_labels[:, i])
        classifiers.append(clf)

    # STEP 6: OPTIMIZE THRESHOLDS
    print("\nSTEP 6: Optimizing thresholds...")

    val_proba = np.array([clf.predict_proba(val_embeddings)[:, 1] for clf in classifiers]).T
    thresholds = {}

    for i, type_name in enumerate(TYPE_NAMES):
        best_f1, best_threshold = 0, 0.5

        for threshold in np.arange(0.2, 0.8, 0.05):
            predictions = (val_proba[:, i] > threshold).astype(int)
            f1 = f1_score(val_labels[:, i], predictions, zero_division=0)
            if f1 > best_f1:
                best_f1, best_threshold = f1, threshold

        thresholds[type_name] = best_threshold
        print(f"  {type_name.capitalize():<12s}: {best_threshold:.2f} (F1={best_f1:.2f})")

    # STEP 7: EVALUATE
    print("\nSTEP 7: Validation performance...")
    for i, type_name in enumerate(TYPE_NAMES):
        predictions = (val_proba[:, i] > thresholds[type_name]).astype(int)
        acc = accuracy_score(val_labels[:, i], predictions)
        f1 = f1_score(val_labels[:, i], predictions, zero_division=0)
        print(f"  {type_name.capitalize():<12s}: Acc={acc:.2%}, F1={f1:.2f}")

    # STEP 8: SAVE
    print("\nSTEP 8: Saving...")
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump({'classifiers': classifiers, 'thresholds': thresholds}, f)
    print(f"✓ Saved to {MODEL_PATH}")

# ============================================================================
# APPLY THRESHOLD CORRECTIONS
# ============================================================================

print("\n" + "="*80)
print("APPLYING THRESHOLD CORRECTIONS")
print("="*80)

original_thresholds = thresholds.copy()

# Fix 1: Raise Lexical threshold to reduce false positives
thresholds['lexical'] = 0.60

# Fix 2: Lower Referential threshold to catch more cases
thresholds['referential'] = 0.45

# Fix 3: Lower Syntactic threshold to catch more cases
thresholds['syntactic'] = 0.55

print("\nThreshold Adjustments:")
print(f"  Lexical:      {original_thresholds['lexical']:.2f} → 0.65 (reduce false positives)")
print(f"  Referential:  {original_thresholds['referential']:.2f} → 0.45 (catch more cases)")
print(f"  Syntactic:    {original_thresholds['syntactic']:.2f} → 0.55 (catch more cases)")
print(f"  Semantic:     {original_thresholds['semantic']:.2f} (unchanged)")
print(f"  Pragmatic:    {original_thresholds['pragmatic']:.2f} (unchanged)")

print("\n✓ Corrected thresholds: {")
for k, v in thresholds.items():
    print(f"    '{k}': {v:.2f},")
print("}")

print("="*80)


IMPROVED AI MODEL - Final Version with Threshold Corrections
Training new model...

STEP 1: Loading data...
Total ambiguous samples: 1336

Original distribution:
  Lexical     :  248 (18.6%)
  Syntactic   :  200 (15.0%)
  Semantic    : 1181 (88.4%)
  Pragmatic   :  335 (25.1%)
  Referential :  266 (19.9%)

STEP 2: Balancing data...
Balanced size: 694

STEP 3: Splitting...
  Train: 555, Val: 139

STEP 4: Generating embeddings...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


STEP 5: Training classifiers...
  Lexical...
  Syntactic...
  Semantic...
  Pragmatic...
  Referential...

STEP 6: Optimizing thresholds...
  Lexical     : 0.35 (F1=0.60)
  Syntactic   : 0.60 (F1=0.80)
  Semantic    : 0.45 (F1=0.93)
  Pragmatic   : 0.50 (F1=0.64)
  Referential : 0.50 (F1=0.93)

STEP 7: Validation performance...
  Lexical     : Acc=58.27%, F1=0.60
  Syntactic   : Acc=87.77%, F1=0.80
  Semantic    : Acc=89.93%, F1=0.93
  Pragmatic   : Acc=69.78%, F1=0.64
  Referential : Acc=94.24%, F1=0.93

STEP 8: Saving...
✓ Saved to /content/drive/MyDrive/improved_ambiguity_model_2.pkl

APPLYING THRESHOLD CORRECTIONS

Threshold Adjustments:
  Lexical:      0.35 → 0.65 (reduce false positives)
  Referential:  0.50 → 0.45 (catch more cases)
  Syntactic:    0.60 → 0.55 (catch more cases)
  Semantic:     0.45 (unchanged)
  Pragmatic:    0.50 (unchanged)

✓ Corrected thresholds: {
    'lexical': 0.60,
    'syntactic': 0.55,
    'semantic': 0.45,
    'pragmatic': 0.50,
    'referential': 0

In [ ]:


# ============================================================================
# SCORING FUNCTION
# ============================================================================

def calculate_ai_ambiguity_score(text, is_ambiguous, confidence):
    """Scoring with corrected thresholds"""

    if not is_ambiguous:
        return {
            'ambiguity_score': 0.0, 'severity': 'CLEAR',
            'justification': 'Classified as unambiguous.',
            'types_detected': [], 'type_scores': {},
            'clarity_scores': {}, 'ai_signals': {}
        }

    embedding = embedder.encode([text])
    predictions = np.array([clf.predict_proba(embedding)[:, 1] for clf in classifiers]).flatten()

    type_scores = {TYPE_NAMES[i]: float(predictions[i]) for i in range(5)}

    # Use CORRECTED thresholds
    detected_types = [TYPE_NAMES[i] for i in range(5) if predictions[i] > thresholds[TYPE_NAMES[i]]]

    if detected_types:
        avg_type_score = np.mean([predictions[i] for i in range(5) if TYPE_NAMES[i] in detected_types])
    else:
        avg_type_score = np.mean(predictions)

    final_score = 0.70 * avg_type_score + 0.30 * confidence
    final_score = max(0.0, min(1.0, final_score))

    if detected_types:
        justification = f"Detected {len(detected_types)} type(s): {', '.join([t.capitalize() for t in detected_types])}"
    else:
        top_idx = np.argmax(predictions)
        justification = f"Weak signals (highest: {TYPE_NAMES[top_idx].capitalize()} at {predictions[top_idx]:.0%}, needs {thresholds[TYPE_NAMES[top_idx]]:.0%})"

    if avg_type_score > 0.6:
        justification += " - high clarity deficit"
    justification += "."

    severity = 'LOW' if final_score < 0.4 else 'MEDIUM' if final_score < 0.7 else 'HIGH'

    return {
        'ambiguity_score': round(final_score, 2),
        'severity': severity,
        'justification': justification,
        'types_detected': detected_types,
        'type_scores': {k: round(v, 2) for k, v in type_scores.items()},
        'type_thresholds': {k: round(v, 2) for k, v in thresholds.items()},
        'clarity_scores': {
            'not_specific': round(min(avg_type_score * 0.8, 1.0), 2),
            'not_measurable': round(min(avg_type_score * 0.9, 1.0), 2),
            'not_testable': round(min(avg_type_score * 0.85, 1.0), 2)
        },
        'ai_signals': {
            'avg_type_score': round(float(avg_type_score), 2),
            'num_detected': len(detected_types)
        }
    }

print("✓ Scoring function ready with corrected thresholds")

✓ Scoring function ready with corrected thresholds


In [ ]:
# ============================================================================
# CELL 14.5: LOAD YOUR BINARY CLASSIFICATION MODEL
# ============================================================================

import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = '/content/drive/MyDrive/FYP_ambiguity'  # Your binary model path

print("="*80)
print("LOADING YOUR BINARY CLASSIFICATION MODEL")
print("="*80)

try:
    # Load tokenizer and model
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    model = RobertaForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)
    model.to(device)
    model.eval()
    print(f"✓ Binary model loaded from: {MODEL_PATH}")
    print(f"✓ Device: {device}")

except Exception as e:
    print(f"Error loading from {MODEL_PATH}: {e}")
    print("\nTrying checkpoint method...")

    # Fallback: Load from checkpoint
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

    checkpoint_path = os.path.join(MODEL_PATH, 'best_model_checkpoint.pt')
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f"✓ Model loaded from checkpoint")
    print(f"✓ Device: {device}")

print("✓ Binary classification model ready")

LOADING YOUR BINARY CLASSIFICATION MODEL
✓ Binary model loaded from: /content/drive/MyDrive/FYP_ambiguity
✓ Device: cuda
✓ Binary classification model ready


In [ ]:
# ============================================================================
# CELL 15: INTERACTIVE PREDICTION WITH SETFIT
# ============================================================================

def predict_ambiguity(text, model, tokenizer, device):
    """Binary prediction using your trained model"""
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][prediction].item()

    return {
        'is_ambiguous': bool(prediction),
        'confidence_score': confidence
    }

print("="*80)
print("INTERACTIVE ANALYSIS - SetFit (Trained on Your Data)")
print("="*80)
print("\nEnter requirements to analyze. Type 'quit' to exit.\n")

while True:
    user_input = input("\nEnter requirement: ").strip()

    if user_input.lower() in ['quit', 'exit', '']:
        print("\nExiting. Goodbye!")
        break

    print("\n" + "="*80)
    print(f"Analyzing: {user_input}")
    print("="*80)

    # Step 1: Binary classification (Your RoBERTa model)
    binary_result = predict_ambiguity(user_input, model, tokenizer, device)
    is_ambiguous = binary_result['is_ambiguous']
    confidence = binary_result['confidence_score']

    print(f"\n📊 BINARY CLASSIFICATION (Your RoBERTa Model):")
    print(f"   Result: {'⚠️  AMBIGUOUS' if is_ambiguous else '✅ CLEAR'}")
    print(f"   Confidence: {confidence:.2%}")

    # Step 2: Type detection and scoring (SetFit model)
    if is_ambiguous:
        print(f"\n🔄 Running SetFit type detection...")
        score_result = calculate_ai_ambiguity_score(user_input, is_ambiguous, confidence)

        print(f"\n🤖 AI AMBIGUITY ANALYSIS (SetFit):")
        print(f"   Score: {score_result['ambiguity_score']:.2f} / 1.0")
        print(f"   Severity: {score_result['severity']}")

        score = score_result['ambiguity_score']
        bar_length = int(score * 40)
        bar = '█' * bar_length + '░' * (40 - bar_length)

        severity_emoji = "🔴" if score_result['severity'] == 'HIGH' else "🟡" if score_result['severity'] == 'MEDIUM' else "🟢"
        print(f"   {severity_emoji} [{bar}] {score:.0%}")

        # Show detected types
        if score_result['types_detected']:
            print(f"\n🎯 DETECTED AMBIGUITY TYPES (Confidence > 50%):")
            for atype in score_result['types_detected']:
                type_score = score_result['type_scores'][atype]
                type_bar = int(type_score * 20)
                type_bar_visual = '█' * type_bar + '░' * (20 - type_bar)
                print(f"   ✓ {atype.capitalize():<12s}: {type_score:.0%} [{type_bar_visual}]")
        else:
            print(f"\n🎯 AMBIGUITY TYPE SIGNALS:")
            print(f"   • No type exceeds 50% confidence threshold")
            # Show top 2 types
            sorted_types = sorted(score_result['type_scores'].items(), key=lambda x: x[1], reverse=True)
            for atype, type_score in sorted_types[:2]:
                print(f"   ○ {atype.capitalize():<12s}: {type_score:.0%}")

        # Show all type scores
        print(f"\n📊 ALL TYPE SCORES:")
        sorted_types = sorted(score_result['type_scores'].items(), key=lambda x: x[1], reverse=True)
        for atype, type_score in sorted_types:
            marker = "✓" if type_score > 0.5 else "○"
            bar_mini = int(type_score * 20)
            mini_bar = '█' * bar_mini + '░' * (20 - bar_mini)
            print(f"   {marker} {atype.capitalize():<12s}: {type_score:.0%} [{mini_bar}]")

        print(f"\n📝 JUSTIFICATION:")
        print(f"   {score_result['justification']}")

        print(f"\n🔍 CLARITY ANALYSIS:")
        for key, value in score_result['clarity_scores'].items():
            status = "⚠️" if value > 0.5 else "✓"
            label = key.replace('_', ' ').title()
            print(f"   {status} {label:<20s}: {value:.0%}")

        # Smart recommendations
        print(f"\n💡 RECOMMENDATION:")
        if score_result['severity'] == 'HIGH':
            print(f"   🔴 CRITICAL: This requirement needs major revision")
            if score_result['types_detected']:
                types_list = ', '.join([t.capitalize() for t in score_result['types_detected']])
                print(f"   → Address these ambiguities: {types_list}")
            print(f"   → Add specific metrics, clear definitions, and testable criteria")
        elif score_result['severity'] == 'MEDIUM':
            print(f"   🟡 MODERATE: This requirement should be clarified")
            if score_result['types_detected']:
                types_list = ', '.join([t.capitalize() for t in score_result['types_detected']])
                print(f"   → Focus on: {types_list}")
            print(f"   → Add measurable thresholds or concrete examples")
        else:
            print(f"   🟢 MINOR: Small improvements would be beneficial")
            print(f"   → Consider adding more precision or specific constraints")
    else:
        print(f"\n✅ CLEAR REQUIREMENT")
        print(f"   This requirement is well-defined and unambiguous.")
        print(f"   No further analysis needed.")

    print("\n" + "="*80)

print("\n" + "="*80)
print("MODELS USED:")
print("  1. Binary Classification: Your trained RoBERTa model")
print("  2. Type Detection: SetFit trained on your labeled data")
print("="*80)

INTERACTIVE ANALYSIS - SetFit (Trained on Your Data)

Enter requirements to analyze. Type 'quit' to exit.


Enter requirement: the system shall be fast

Analyzing: the system shall be fast

📊 BINARY CLASSIFICATION (Your RoBERTa Model):
   Result: ⚠️  AMBIGUOUS
   Confidence: 97.83%

🔄 Running SetFit type detection...


NameError: name 'embedder' is not defined